# Lab 07: Memory, caching and checkpoints (reference solution)

**Time:** about 2 h 30 min.

**Checklist questions this lab gives you evidence for:**<br><span style="color:#d93025"><b>5.1 Executor memory regions</b></span><br><span style="color:#d93025"><b>5.2 Spilling to disk</b></span><br><span style="color:#d93025"><b>5.3 spark.memory.fraction</b></span><br><span style="color:#d93025"><b>5.4 Off-heap memory</b></span><br><span style="color:#d93025"><b>5.5 Driver OOM</b></span><br><span style="color:#d93025"><b>5.6 User memory</b></span><br><span style="color:#d93025"><b>5.7 GC overhead</b></span><br><span style="color:#d93025"><b>5.8 unpersist()</b></span><br><span style="color:#d93025"><b>5.9 Risk of high executor cores</b></span><br><span style="color:#d93025"><b>5.10 Unified memory management</b></span><br><span style="color:#d93025"><b>5.11 cache / persist / checkpoint / localCheckpoint</b></span><br><span style="color:#d93025"><b>6.10 Limit checkpoint()</b></span>

**Goal:** compute the memory regions by hand and find them in the Spark UI, cause and remove a spill, and compare cache, persist and checkpoint. This lab runs Spark with a small 512 MB heap so memory pressure shows quickly. The last section is practice with snapshots, percentiles and sampling.

## Data information
A small online shop. The setup cell below creates three tables in `data/` (seeded, so everyone gets the same rows):

**orders** (`data/orders.parquet`, 1,000,000 rows, January to March 2024)
* *order_id*, *customer_id* (empty for guest checkout), *product_id*, *quantity* (1 to 5)
* *amount*: quantity × product price, in dollars
* *status*: paid, shipped, cancelled or returned; *payment_method*: card, cash or wallet
* *order_ts*, *order_date*: when the order was placed
* *tracking_code*: a 32-character code

**customers** (`data/customers.csv`, 10,000 rows): *customer_id*, *name*, *country*, *signup_date*, *segment* (retail or business). Customer 1 is "Big Reseller Ltd", who places 30% of all orders.

**products** (`data/products.csv`, 200 rows): *product_id*, *name*, *category*, *price*

## How to work
* Write your answer in the code cell under each question, then run the check cell below it: ✅ correct, ❌ try again.
* When a question says *guess first*, write your guess as a comment before you run anything.
* Keep the Spark UI open (the setup cell prints its address).
* Stuck for more than 10 minutes? Read the documentation list at the top, then the reference solution.


## Documentation
Everything you need is on these pages. The questions do not repeat them: read, try things out in a cell, and find the answers yourself.

* [Tuning Spark: memory management](https://spark.apache.org/docs/latest/tuning.html#memory-management-overview)
* [Configuration: memory management](https://spark.apache.org/docs/latest/configuration.html#memory-management)
* [Spark UI: Executors tab](https://spark.apache.org/docs/latest/web-ui.html#executors-tab)
* [Spark UI: Stage detail](https://spark.apache.org/docs/latest/web-ui.html#stages-tab)
* [Tuning Spark: memory tuning](https://spark.apache.org/docs/latest/tuning.html#memory-tuning)
* [RDD persistence](https://spark.apache.org/docs/latest/rdd-programming-guide.html#rdd-persistence)
* [DataFrame.persist](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.persist.html)
* [DataFrame.checkpoint](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.checkpoint.html)
* [Caching data](https://spark.apache.org/docs/latest/sql-performance-tuning.html#caching-data)
* [Catalog.clearCache](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.Catalog.clearCache.html)
* [DataFrame.exceptAll](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.exceptAll.html)
* [Column.eqNullSafe](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.Column.eqNullSafe.html)
* [functions.percentile](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.percentile.html)
* [functions.percentile_approx](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.percentile_approx.html)
* [DataFrame.sampleBy](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.sampleBy.html)
* [Configuration: application properties](https://spark.apache.org/docs/latest/configuration.html#application-properties)
* [DataFrame.toLocalIterator](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.toLocalIterator.html)

## Setup
Run the cell below once (its code is hidden: click it to expand). It creates the data and starts Spark.

In [ ]:
# Setup: run this cell once. It creates the shop data in data/ and starts Spark.
# Nothing to solve here, and you do not need to read it.
import contextlib, hashlib, io, json, os, re, shutil, sys, time, urllib.request
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)   # Spark's Python workers use this kernel's Python
from pyspark.sql import SparkSession, Window, functions as F

# ---------------------------------------------------------------- the shop data ---------
# A small online shop, generated from seeded hash functions: every machine gets the same rows.
COUNTRIES = ["Uzbekistan", "Germany", "USA", "India", "Brazil", "Japan"]
CATEGORIES = ["books", "toys", "garden", "sports", "kitchen", "beauty", "music", "office"]
N_CUSTOMERS, N_PRODUCTS, N_ORDERS = 10_000, 200, 1_000_000

def _u(col, salt):
    """Pseudo-random number in [0, 1) computed from a column (same result on every machine)."""
    return F.pmod(F.xxhash64(col, F.lit(salt)), F.lit(1_000_000)) / 1_000_000.0

def _pick(u, options):
    return F.element_at(F.array(*[F.lit(o) for o in options]), (F.floor(u * len(options)) + 1).cast("int"))

def make_customers(spark):
    c = spark.range(1, N_CUSTOMERS + 1).withColumnRenamed("id", "customer_id")
    return c.select(
        "customer_id",
        F.when(F.col("customer_id") == 1, "Big Reseller Ltd")
         .otherwise(F.concat(F.lit("Customer "), F.col("customer_id").cast("string"))).alias("name"),
        _pick(_u(F.col("customer_id"), 1), COUNTRIES).alias("country"),
        F.date_add(F.lit("2022-01-01").cast("date"), F.floor(_u(F.col("customer_id"), 2) * 730).cast("int")).alias("signup_date"),
        F.when((F.col("customer_id") == 1) | (_u(F.col("customer_id"), 3) < 0.2), "business").otherwise("retail").alias("segment"),
    )

def make_products(spark):
    p = spark.range(1, N_PRODUCTS + 1).withColumnRenamed("id", "product_id")
    return p.select(
        "product_id",
        F.concat(F.lit("Product "), F.col("product_id").cast("string")).alias("name"),
        F.element_at(F.array(*[F.lit(c) for c in CATEGORIES]), (F.col("product_id") % 8 + 1).cast("int")).alias("category"),
        F.round((F.col("product_id") * 37) % 195 + 5.99, 2).alias("price"),
    )

def make_orders(spark, n=N_ORDERS, dirty=False):
    i = F.col("id")
    customer = (F.when(_u(i, 11) < 0.02, F.lit(None).cast("long"))                    # guest checkout: no customer
                 .when(_u(i, 10) < 0.30, F.lit(1).cast("long"))                       # the big reseller: 30% of orders
                 .otherwise((F.floor(_u(i, 12) * (N_CUSTOMERS - 1)) + 2).cast("long")))
    quantity = F.when(customer == 1, F.lit(5)).otherwise(F.floor(_u(i, 13) * 5) + 1).cast("int")   # the reseller always orders in bulk
    product = (F.floor(_u(i, 14) * N_PRODUCTS) + 1).cast("int")
    ts = F.lit("2024-01-01 00:00:00").cast("timestamp") + F.make_interval(secs=F.floor(_u(i, 15) * 91 * 86400).cast("int"))
    u = _u(i, 17)
    status = F.when(u < 0.55, "paid").when(u < 0.80, "shipped").when(u < 0.92, "cancelled").otherwise("returned")
    v = _u(i, 18)
    payment = F.when(v < 0.60, "card").when(v < 0.85, "cash").otherwise("wallet")
    df = spark.range(0, n, 1, 4).select(
        (i + 1).alias("order_id"), customer.alias("customer_id"), product.alias("product_id"), quantity.alias("quantity"),
        F.round(quantity * (((product * 37) % 195) + 5.99), 2).alias("amount"),
        status.alias("status"), payment.alias("payment_method"), ts.alias("order_ts"), F.md5(i.cast("string")).alias("tracking_code"))
    df = df.withColumn("order_date", F.to_date("order_ts"))
    if dirty:    # the monthly files of the final project contain mistakes on purpose
        df = df.withColumn("amount", F.when(_u(F.col("order_id"), 19) < 0.008, -F.col("amount")).otherwise(F.col("amount")))
        df = df.withColumn("quantity", F.when(_u(F.col("order_id"), 20) < 0.006, 0).otherwise(F.col("quantity")))
        df = df.unionByName(df.filter(_u(F.col("order_id"), 21) < 0.005))              # exact duplicates
    return df.select("order_id", "customer_id", "product_id", "quantity", "amount", "status",
                     "payment_method", "order_ts", "order_date", "tracking_code")

def _create_data(spark):
    os.makedirs("data/monthly", exist_ok=True)
    make_customers(spark).toPandas().to_csv("data/customers.csv", index=False)
    make_products(spark).toPandas().to_csv("data/products.csv", index=False)
    make_orders(spark).repartition(4).write.mode("overwrite").parquet("data/orders.parquet")
    dirty = make_orders(spark, dirty=True)
    wrong = _u(F.col("order_id"), 22) < 0.003          # a few orders land in the wrong month's file
    month = F.month("order_date")
    file_month = F.when(wrong, month % 3 + 1).otherwise(month)
    for m in (1, 2, 3):
        (dirty.filter(file_month == m).coalesce(1).write.mode("overwrite")
              .parquet(f"data/monthly/orders_2024-{m:02d}.parquet"))

# ---------------------------------------------------------------- the Spark session ----
def new_session(master="local[4]", driver_memory="2g", delta=False, avro=False, aqe=False, conf=None):
    builder = (SparkSession.builder.master(master).appName("day07_lab")
               .config("spark.driver.memory", driver_memory)
               .config("spark.ui.showConsoleProgress", "false")
               .config("spark.sql.adaptive.enabled", str(aqe).lower())   # off = exact job and stage counts
               .config("spark.sql.session.timeZone", "UTC")                # same dates and hours for everyone
               .config("spark.sql.warehouse.dir", os.path.abspath("spark-warehouse"))
               # Spark 4.2.0 bug (SPARK-60294): Python UDFs on Parquet can hang without this.
               .config("spark.sql.execution.arrow.pythonUDF.columnarInput.enabled", "false"))
    for k, v in (conf or {}).items():
        builder = builder.config(k, v)
    if delta:
        from delta import configure_spark_with_delta_pip
        builder = (builder.config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
                          .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))
        builder = configure_spark_with_delta_pip(builder, extra_packages=["org.apache.spark:spark-avro_2.13:4.2.0"] if avro else None)
    elif avro:
        builder = builder.config("spark.jars.packages", "org.apache.spark:spark-avro_2.13:4.2.0")
    s = builder.getOrCreate()
    s.sparkContext.setLogLevel("ERROR")
    return s

for _d in ("spark-warehouse", "metastore_db"):
    shutil.rmtree(_d, ignore_errors=True)               # a clean catalog on every start
spark = new_session(driver_memory="512m")
if not all(os.path.exists(f) for f in ("data/orders.parquet", "data/customers.csv", "data/products.csv", "data/monthly/orders_2024-03.parquet")):
    print("creating the shop data (about 30 seconds, once)...")
    _create_data(spark)
print(f"Spark {spark.version} | UI: {spark.sparkContext.uiWebUrl}")

# ---------------------------------------------------------------- helpers ----------------
def _tracker():
    return spark.sparkContext.statusTracker()

def _run_group(action):
    sc = spark.sparkContext
    group = f"g{time.time_ns()}"
    sc.setJobGroup(group, group)
    try:
        action()
    finally:
        sc.setLocalProperty("spark.jobGroup.id", None)
    return [_tracker().getJobInfo(j) for j in sorted(_tracker().getJobIdsForGroup(group))]

def jobs_run(action):
    """How many Spark jobs calling action() starts."""
    return len(_run_group(action))

def stages_run(action):
    """How many stages those jobs had in total."""
    return sum(len(j.stageIds) for j in _run_group(action))

def last_stage_tasks(action):
    """Number of tasks in the last stage action() ran."""
    jobs = _run_group(action)
    return _tracker().getStageInfo(max(s for j in jobs for s in j.stageIds)).numTasks

def _api(path):
    sc = spark.sparkContext
    return json.load(urllib.request.urlopen(f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}{path}"))

api = _api   # read anything from the Spark UI's REST API, e.g. api("/executors")

def _stage_rows(action):
    jobs = _run_group(action)
    ids = {s for j in jobs for s in j.stageIds}
    return [s for s in _api("/stages?length=2000") if s["stageId"] in ids]

def shuffle_bytes(action):
    """Total shuffle bytes written by the stages action() ran."""
    return sum(s["shuffleWriteBytes"] for s in _stage_rows(action))

def spill_bytes(action):
    """(memory, disk) bytes spilled by the stages action() ran."""
    rows = _stage_rows(action)
    return sum(s["memoryBytesSpilled"] for s in rows), sum(s["diskBytesSpilled"] for s in rows)

def skew_ratio(action):
    """Slowest task / median task, in the stage with the most tasks that action() ran."""
    rows = _stage_rows(action)
    st = max(rows, key=lambda s: s["numTasks"])
    tasks = _api(f"/stages/{st['stageId']}/{st['attemptId']}/taskList?length=5000")
    times = sorted(t["taskMetrics"]["executorRunTime"] for t in tasks if t.get("taskMetrics"))
    return round(times[-1] / max(times[len(times) // 2], 1), 1)

def failed_attempts():
    """Failed task attempts in the most recent failed stage (from the Spark UI's REST API)."""
    failed = _api("/stages?status=failed")
    if not failed:
        return 0
    st = max(failed, key=lambda s: s["stageId"])
    tasks = _api(f"/stages/{st['stageId']}/{st['attemptId']}/taskList?length=500")
    return sum(1 for t in tasks if t["status"] == "FAILED")

def sql_metric(action, metric):
    """Value of one scan metric (for example 'number of files read') of the query action() runs."""
    desc = f"q{time.time_ns()}"
    spark.sparkContext.setJobDescription(desc)
    try:
        action()
    finally:
        spark.sparkContext.setJobDescription(None)
    for ex in _api("/sql?details=true&planDescription=false&length=2000"):
        if ex.get("description") == desc:
            vals = [m["value"] for n in ex["nodes"] for m in n["metrics"] if m["name"] == metric]
            return vals[0] if vals else None

def noop(df):
    """Run the whole query but throw the result away."""
    df.write.format("noop").mode("overwrite").save()

def plan(df, mode="formatted"):
    """explain() output as a string."""
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        df.explain(mode)
    return buf.getvalue()

def count_nodes(df, *names):
    """How often each operator name appears in the simple plan."""
    text = plan(df, "simple")
    return {n: len(re.findall(r"\b" + re.escape(n) + r"\b", text)) for n in names}

def optimized(df):
    """The optimized logical plan as a string."""
    return df._jdf.queryExecution().optimizedPlan().toString()

def operators(df):
    """Operator names of the physical plan tree, top to bottom."""
    tree = plan(df).split("\n\n")[0].splitlines()[1:]
    return [re.sub(r"\s*\(\d+\)$", "", line).strip(" +-*:") for line in tree]

def rules_fired(df):
    """Names of the Catalyst rules that changed this query's plan."""
    qe = df._jdf.queryExecution(); qe.executedPlan()
    it, fired = qe.tracker().rules().iterator(), set()
    while it.hasNext():
        kv = it.next()
        if kv._2().numEffectiveInvocations() > 0:
            fired.add(kv._1().split(".")[-1].split("$")[-1])
    return fired

def estimate(df):
    """(sizeInBytes, rowCount or None) the optimizer estimates for df."""
    stats = df._jdf.queryExecution().optimizedPlan().stats()
    return int(str(stats.sizeInBytes())), (int(str(stats.rowCount().get())) if stats.rowCount().isDefined() else None)

def partition_sizes(df):
    """Rows per partition, largest first."""
    return sorted((r["count"] for r in df.groupBy(F.spark_partition_id().alias("p")).count().collect()), reverse=True)

def timeit(action):
    start = time.perf_counter(); action(); return round(time.perf_counter() - start, 2)

def folder_mb(path):
    files = [os.path.join(d, f) for d, _, fs in os.walk(path) for f in fs] if os.path.isdir(path) else [path]
    return round(sum(os.path.getsize(f) for f in files) / 1e6, 1)

# ---------------------------------------------------------------- checks -----------------
_EXPECTED = {
 "7.1": "0ffcc981a2262ae6",
 "7.4": "58ef262ba45715aa",
 "7.5": "9749d682cc49fcd5",
 "7.6": "e026dde8fb686a83",
 "7.7": "595e5dcc95ef4a8f",
 "7.8": "78e5bdbb955cf1be",
 "7.10b": "ce2ab54517e29104"
}
_RECORD = os.environ.get("LAB_RECORD")

def _norm(v):
    if isinstance(v, bool) or v is None: return v
    if isinstance(v, int): return v
    if isinstance(v, float): return round(v, 2)
    if isinstance(v, str): return v.strip()
    if isinstance(v, dict): return {str(k): _norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
    if isinstance(v, (list, tuple)): return [_norm(x) for x in v]
    if isinstance(v, (set, frozenset)): return sorted((_norm(x) for x in v), key=json.dumps)
    if hasattr(v, "item"): return _norm(v.item())
    if type(v).__name__ == "Decimal": return round(float(v), 2)
    raise TypeError(f"check() needs plain Python values, got {type(v).__name__}")

def check(task, value):
    normal = _norm(value)
    if _RECORD:
        rec = json.load(open(_RECORD)) if os.path.exists(_RECORD) else {}
        rec[task] = normal; json.dump(rec, open(_RECORD, "w"))
    ok = hashlib.sha256(json.dumps(normal).encode()).hexdigest()[:16] == _EXPECTED.get(task)
    print(f"✅ {task} correct" if ok else f"❌ {task} not yet: check your code and the question again")

def check_true(task, condition, hint=""):
    print(f"✅ {task} correct" if condition else f"❌ {task} not yet. {hint}")


Load the tables the way Lab 01 taught you (given):

In [ ]:
orders = spark.read.parquet("data/orders.parquet")
customers = (spark.read.option("header", True)
             .schema("customer_id long, name string, country string, signup_date date, segment string")
             .csv("data/customers.csv"))
products = (spark.read.option("header", True)
            .schema("product_id long, name string, category string, price double")
            .csv("data/products.csv"))
for name, df in [("orders", orders), ("customers", customers), ("products", products)]:
    df.createOrReplaceTempView(name)   # so SQL can use the tables too

## A. The memory regions

**7.1 Compute the executor memory regions by hand.** Write a function `regions(heap_mb)` that returns `(usable, unified, protected_storage, user)` in MB, rounded to 1 decimal: 300 MB are reserved; *unified* = usable × `spark.memory.fraction` (0.6); *protected_storage* = unified × `spark.memory.storageFraction` (0.5); *user* = the rest of usable. Put the results for heaps of 4096, 10240 and 16384 MB in a dict *table* (`{heap_mb: regions(heap_mb)}`).

In [ ]:
def regions(heap_mb):
    usable = heap_mb - 300
    unified = usable * 0.6
    return (round(usable, 1), round(unified, 1), round(unified * 0.5, 1), round(usable - unified, 1))

table = {h: regions(h) for h in (4096, 10240, 16384)}
table

In [ ]:
check("7.1", table)

**7.2 Find the unified memory in the Spark UI.** This session has a 512 MB heap. Read the executor's *maxMemory* from the REST API (`api("/executors")`, in the Executors tab it is called Storage Memory) in MiB, rounded to 1 decimal (*storage_mib*). Compare it with your `regions(512)`. Which region does it show?

In [ ]:
executors = api("/executors")
storage_mib = round(executors[0]["maxMemory"] / 2**20, 1)
storage_mib, regions(512)

In [ ]:
check_true("7.2", abs(storage_mib - regions(512)[1]) < 10, "the executor shows the whole unified region (storage + execution), not only storage")

## B. Spilling

**7.3 Cause a spill, then remove it.** Sort the 2 million rows of `big = orders.union(orders)` by *tracking_code* and write with `noop`, once with `spark.sql.shuffle.partitions` = 2 and once with 200. Measure `spill_bytes` for both: *spill_2* and *spill_200* are `(memory, disk)` tuples. Find Spill (Memory) and Spill (Disk) in the stage page of the Spark UI. Reset the setting to 200.

In [ ]:
big = orders.union(orders)
spark.conf.set("spark.sql.shuffle.partitions", 2)
spill_2 = spill_bytes(lambda: noop(big.orderBy("tracking_code")))
spark.conf.set("spark.sql.shuffle.partitions", 200)
spill_200 = spill_bytes(lambda: noop(big.orderBy("tracking_code")))
spill_2, spill_200

In [ ]:
check_true("7.3", spill_2[1] > 0 and spill_200 == (0, 0), "2 huge partitions must spill; 200 small ones must not")

## C. cache, persist, checkpoint

**7.4 Cache the right thing.** `joined = orders.join(customers, "customer_id")` is used by three reports. Time all three together (`reports(df)`, given) three times: with no persistence (*t_none*), after `joined.cache()` (*t_cache_first* fills the cache, *t_cache_second* reads it), and after `unpersist()` and `persist(StorageLevel.DISK_ONLY)` (*t_disk*, second run). Read the Storage tab through `api("/storage/rdd")` and put `(storageLevel, numCachedPartitions, numPartitions)` of the cached DataFrame in *storage_info* (taken while it is cached with `cache()`). Free it with `unpersist()` at the end.

In [ ]:
from pyspark import StorageLevel

def reports(df):
    df.groupBy("country").count().collect()
    df.groupBy("segment").agg(F.sum("amount")).collect()
    df.groupBy("status").agg(F.avg("amount")).collect()

joined = orders.join(customers, "customer_id")
t_none = timeit(lambda: reports(joined))
joined.cache()
t_cache_first = timeit(lambda: reports(joined))
t_cache_second = timeit(lambda: reports(joined))
rdd = api("/storage/rdd")[0]
storage_info = (rdd["storageLevel"], rdd["numCachedPartitions"], rdd["numPartitions"])
joined.unpersist(blocking=True)
joined.persist(StorageLevel.DISK_ONLY)
reports(joined)
t_disk = timeit(lambda: reports(joined))
joined.unpersist(blocking=True)
storage_info, t_none, t_cache_first, t_cache_second, t_disk

In [ ]:
check("7.4", storage_info)
check_true("7.4b", t_cache_second < t_none and t_cache_first > t_cache_second, "the second cached run is the fastest; the first one pays for filling the cache")

**7.5 Cut a long lineage.** Build *df* from `spark.range(1000)` with a loop of 30 `withColumn(f"c{i}", F.col("id") + i)` steps. Set the checkpoint folder (`spark.sparkContext.setCheckpointDir("checkpoints")`) and create `df.checkpoint()` (*reliable*) and `df.localCheckpoint()` (*local*). The length of `plan(df, "extended")` is much shorter after a checkpoint: *shorter* = True if the checkpointed plan is less than a third of the original. Are there files in `checkpoints/` (*has_files*)?

In [ ]:
df = spark.range(1000)
for i in range(30):
    df = df.withColumn(f"c{i}", F.col("id") + i)
spark.sparkContext.setCheckpointDir("checkpoints")
reliable = df.checkpoint()
local = df.localCheckpoint()
shorter = len(plan(reliable, "extended")) < len(plan(df, "extended")) / 3
has_files = any(files for _, _, files in os.walk("checkpoints"))
shorter, has_files

In [ ]:
check("7.5", [shorter, has_files])

**7.6 A stale cache.** Save *customers* as Parquet in `data/cust_copy`, read it as *p* and cache it (`p.cache().count()`). Call `add_external_row()` (given: another application adds a file). A fresh `spark.read.parquet("data/cust_copy").count()` is *count_stale*. Then clear the cache with `spark.catalog.clearCache()` and count again (*count_fresh*). Put `[count_stale, count_fresh]` in *result*.

In [ ]:
import datetime
import pyarrow as pa
import pyarrow.parquet as pq

def add_external_row():
    """Another application adds a file to the folder (given)."""
    pq.write_table(pa.table({"customer_id": pa.array([999999], pa.int64()), "name": ["Outsider"], "country": ["Japan"],
                             "signup_date": pa.array([datetime.date(2024, 1, 1)], pa.date32()), "segment": ["retail"]}),
                   "data/cust_copy/part-external.parquet")

shutil.rmtree("data/cust_copy", ignore_errors=True)
customers.write.parquet("data/cust_copy")
p = spark.read.parquet("data/cust_copy")
p.cache().count()
add_external_row()
count_stale = spark.read.parquet("data/cust_copy").count()
spark.catalog.clearCache()
count_fresh = spark.read.parquet("data/cust_copy").count()
result = [count_stale, count_fresh]
result

In [ ]:
check("7.6", result)

## D. Coding practice: snapshots, percentiles and samples

Three problems. Write them from scratch.

**7.7 What changed between two snapshots?** *yesterday* and *today* (given) list customers. Find the ids that were added (*added*), removed (*removed*) and changed (same *id*, different *name* or *segment*; *changed*), each as a sorted list of ids. Put `[added, removed, changed]` in *result*. Use a full outer join, or `exceptAll`.

In [ ]:
yesterday = spark.createDataFrame([(1, "Ann", "retail"), (2, "Bob", "retail"), (3, "Cy", "business"), (4, "Di", "retail")], "id int, name string, segment string")
today = spark.createDataFrame([(1, "Ann", "retail"), (2, "Bob", "business"), (4, "Di", "retail"), (5, "Eve", "retail")], "id int, name string, segment string")

j = yesterday.alias("y").join(today.alias("t"), "id", "full")
added = sorted(r["id"] for r in j.filter(F.col("y.name").isNull()).collect())
removed = sorted(r["id"] for r in j.filter(F.col("t.name").isNull()).collect())
changed = sorted(r["id"] for r in j.filter(F.col("y.name").isNotNull() & F.col("t.name").isNotNull() & (~F.col("y.name").eqNullSafe(F.col("t.name")) | ~F.col("y.segment").eqNullSafe(F.col("t.segment")))).collect())
result = [added, removed, changed]
result

In [ ]:
check("7.7", result)

**7.8 What is the median and the 90th percentile of the order amount per *status*?** Use the exact `F.percentile` and round to 2 decimals. *percentiles* = dict `{status: (median, p90)}`.

In [ ]:
rows = orders.groupBy("status").agg(F.percentile("amount", 0.5).alias("p50"), F.percentile("amount", 0.9).alias("p90")).collect()
percentiles = {r["status"]: (round(r["p50"], 2), round(r["p90"], 2)) for r in rows}
percentiles

In [ ]:
check("7.8", percentiles)

**7.9 Take a stratified sample.** With `sampleBy` (column *status*, seed 42) keep about 1% of the *paid*, 2% of the *shipped*, 5% of the *cancelled* and 10% of the *returned* orders. *sample_counts* = dict `{status: rows in the sample}`. They are only approximately the fractions: *close* = True if every count is within 10% of its expected value.

In [ ]:
fractions = {"paid": 0.01, "shipped": 0.02, "cancelled": 0.05, "returned": 0.10}
sample = orders.sampleBy("status", fractions, seed=42)
sample_counts = {r["status"]: r["count"] for r in sample.groupBy("status").count().collect()}
full = {r["status"]: r["count"] for r in orders.groupBy("status").count().collect()}
close = all(abs(sample_counts[s] - full[s] * f) <= 0.10 * full[s] * f for s, f in fractions.items())
close

In [ ]:
check_true("7.9", close, "each sample count should be within 10% of fraction x rows")

## E. Driver out of memory

**7.10 Make the driver run out of result memory, then fix it.** Restart Spark with a small result limit: `spark.stop()`, then `spark = new_session(driver_memory="512m", conf={"spark.driver.maxResultSize": "20m"})`, and load *orders* again. Try `orders.collect()` and catch the error: *limit_error* = True if the message mentions `maxResultSize`. Fix it without raising the limit: collect only an aggregate (the number of orders per *status*, *status_counts* as a dict) and iterate over all rows with `toLocalIterator()` to count them (*streamed*).

In [ ]:
spark.stop()
spark = new_session(driver_memory="512m", conf={"spark.driver.maxResultSize": "20m"})
orders = spark.read.parquet("data/orders.parquet")
try:
    orders.collect()
    limit_error = False
except Exception as e:
    limit_error = "maxResultSize" in str(e)
status_counts = {r["status"]: r["count"] for r in orders.groupBy("status").count().collect()}
streamed = sum(1 for _ in orders.select("order_id").toLocalIterator())
limit_error, status_counts, streamed

In [ ]:
check_true("7.10", limit_error, "collecting 1,000,000 rows should exceed the 20 MB result limit")
check("7.10b", [status_counts, streamed])

---
Done? In the Theory part of Day 7, answer the red checklist topics aloud, with the numbers from this lab as evidence.